# GUI Element Detection — Model Comparison

Comparative evaluation of all trained models:
- **YOLOv8n** — single-stage, 30 epochs, full dataset
- **YOLOv10n** — single-stage, 30 epochs, full dataset
- **v26 (yolo26n)** — single-stage, early stopping, full dataset
- **Faster R-CNN** (ResNet-50 + FPN) — two-stage, 8 epochs, 5% subset

**Metrics evaluated:** Precision, Recall, F1-Score, mAP@50, mAP@50-95


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

# ── Paths (relative to this notebook's location) ──────────────────────────────
BASE = os.path.abspath("../")

YOLO_RUNS = {
    "YOLOv8n":  os.path.join(BASE, "models/yolo/runs/detect/yolov8"),
    "YOLOv10n": os.path.join(BASE, "models/yolo/runs/detect/yolov10"),
    "v26":      os.path.join(BASE, "models/yolo/runs/detect/yolo26"),
}
FRCNN_CSV  = os.path.join(BASE, "models/faster-rcnn/results_fasterrcnn.csv")
FRCNN_PTH  = os.path.join(BASE, "models/faster-rcnn/best_fasterrcnn.pth")
TEST_IMAGES = os.path.join(BASE, "data/unified/combined/test/images")

CLASS_NAMES = ["Button", "Text", "Image", "Icon", "Input", "Link",
               "Checkbox", "Toggle", "Toolbar", "Navigation", "Modal", "Tab"]

print("Base path:", BASE)
print("YOLO runs found:", {k: os.path.isdir(v) for k, v in YOLO_RUNS.items()})
print("Faster R-CNN CSV exists:", os.path.isfile(FRCNN_CSV))


## 1. Load Training Results

In [ ]:
def load_yolo_csv(run_dir):
    csv_path = os.path.join(run_dir, "results.csv")
    if not os.path.isfile(csv_path):
        return None
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        "metrics/precision(B)": "precision",
        "metrics/recall(B)":    "recall",
        "metrics/mAP50(B)":     "map50",
        "metrics/mAP50-95(B)":  "map50_95",
    })
    df["f1"] = 2 * df["precision"] * df["recall"] / (df["precision"] + df["recall"] + 1e-9)
    return df


def load_frcnn_csv(csv_path):
    if not os.path.isfile(csv_path):
        return None
    df = pd.read_csv(csv_path)
    if df.empty:
        return None
    df = df.rename(columns={"map50": "map50", "recall50": "recall", "map50_95": "map50_95"})
    # Faster R-CNN CSV has no precision column — add placeholder
    if "precision" not in df.columns:
        df["precision"] = float("nan")
    df["f1"] = 2 * df["precision"] * df["recall"] / (df["precision"] + df["recall"] + 1e-9)
    return df


# Load all
yolo_dfs = {name: load_yolo_csv(path) for name, path in YOLO_RUNS.items()}
frcnn_df  = load_frcnn_csv(FRCNN_CSV)

for name, df in yolo_dfs.items():
    status = f"{len(df)} epochs" if df is not None else "NOT FOUND"
    print(f"  {name}: {status}")

frcnn_status = f"{len(frcnn_df)} epochs" if frcnn_df is not None else "NOT FOUND / empty"
print(f"  Faster R-CNN: {frcnn_status}")


## 2. Best-Epoch Metrics (Comparison Table)

In [ ]:
def best_row(df, metric="map50"):
    if df is None or df.empty:
        return None
    return df.loc[df[metric].idxmax()]


rows = []
for name, df in yolo_dfs.items():
    row = best_row(df)
    if row is not None:
        rows.append({
            "Model":      name,
            "Best Epoch": int(row.get("epoch", 0)),
            "Precision":  round(float(row["precision"]), 4),
            "Recall":     round(float(row["recall"]),    4),
            "F1-Score":   round(float(row["f1"]),        4),
            "mAP@50":     round(float(row["map50"]),     4),
            "mAP@50-95":  round(float(row["map50_95"]),  4),
        })

frcnn_row = best_row(frcnn_df)
if frcnn_row is not None:
    rows.append({
        "Model":      "Faster R-CNN",
        "Best Epoch": int(frcnn_row.get("epoch", 0)),
        "Precision":  round(float(frcnn_row["precision"]), 4) if not pd.isna(frcnn_row["precision"]) else "N/A",
        "Recall":     round(float(frcnn_row["recall"]),    4),
        "F1-Score":   round(float(frcnn_row["f1"]),        4) if not pd.isna(frcnn_row["f1"]) else "N/A",
        "mAP@50":     round(float(frcnn_row["map50"]),     4),
        "mAP@50-95":  round(float(frcnn_row["map50_95"]),  4),
    })

summary_df = pd.DataFrame(rows).set_index("Model")
print("\n=== Best-Epoch Comparison Table ===")
print(summary_df.to_string())
summary_df


## 3. Training Curves

In [ ]:
COLORS = {"YOLOv8n": "#2196F3", "YOLOv10n": "#4CAF50", "v26": "#FF9800", "Faster R-CNN": "#9C27B0"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Validation Metrics per Epoch", fontsize=14, fontweight="bold")

metrics = [
    ("map50",    "mAP@50"),
    ("recall",   "Recall"),
    ("map50_95", "mAP@50-95"),
]

for ax, (col, title) in zip(axes, metrics):
    for name, df in yolo_dfs.items():
        if df is not None and col in df.columns:
            ax.plot(df["epoch"], df[col], label=name, color=COLORS[name], linewidth=2)

    if frcnn_df is not None and col in frcnn_df.columns:
        ax.plot(frcnn_df["epoch"], frcnn_df[col],
                label="Faster R-CNN", color=COLORS["Faster R-CNN"],
                linewidth=2, linestyle="--")

    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig("training_curves_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves_comparison.png")


## 4. Final Metrics Bar Chart

In [ ]:
if not summary_df.empty:
    numeric_cols = ["mAP@50", "mAP@50-95", "Recall"]
    plot_df = summary_df[numeric_cols].copy()
    # keep only numeric rows
    for col in numeric_cols:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

    fig, ax = plt.subplots(figsize=(10, 6))
    x      = np.arange(len(plot_df))
    width  = 0.25
    colors = ["#2196F3", "#4CAF50", "#FF5722"]

    for i, col in enumerate(numeric_cols):
        bars = ax.bar(x + i * width, plot_df[col], width, label=col, color=colors[i], alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h):
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                        f"{h:.3f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x + width)
    ax.set_xticklabels(plot_df.index, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Model Comparison — Best Epoch Metrics", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("metrics_bar_chart.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: metrics_bar_chart.png")


## 5. Confusion Matrices (YOLO Models)

In [ ]:
fig, axes = plt.subplots(1, len(YOLO_RUNS), figsize=(18, 6))
fig.suptitle("Normalised Confusion Matrices", fontsize=14, fontweight="bold")

for ax, (name, run_dir) in zip(axes, YOLO_RUNS.items()):
    cm_path = os.path.join(run_dir, "confusion_matrix_normalized.png")
    if os.path.isfile(cm_path):
        img = Image.open(cm_path)
        ax.imshow(img)
        ax.set_title(name, fontsize=12)
        ax.axis("off")
    else:
        ax.text(0.5, 0.5, f"{name}\nNot found", ha="center", va="center",
                transform=ax.transAxes, fontsize=12)
        ax.axis("off")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Precision-Recall Curves (YOLO Models)

In [ ]:
fig, axes = plt.subplots(1, len(YOLO_RUNS), figsize=(18, 6))
fig.suptitle("Precision-Recall Curves", fontsize=14, fontweight="bold")

for ax, (name, run_dir) in zip(axes, YOLO_RUNS.items()):
    pr_path = os.path.join(run_dir, "BoxPR_curve.png")
    if os.path.isfile(pr_path):
        img = Image.open(pr_path)
        ax.imshow(img)
        ax.set_title(name, fontsize=12)
        ax.axis("off")
    else:
        ax.text(0.5, 0.5, f"{name}\nNot found", ha="center", va="center",
                transform=ax.transAxes, fontsize=12)
        ax.axis("off")

plt.tight_layout()
plt.savefig("pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. F1-Score Evolution per Epoch

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, df in yolo_dfs.items():
    if df is not None and "f1" in df.columns:
        ax.plot(df["epoch"], df["f1"], label=name, color=COLORS[name], linewidth=2, marker="o", markersize=3)

if frcnn_df is not None and "f1" in frcnn_df.columns:
    valid = frcnn_df.dropna(subset=["f1"])
    if not valid.empty:
        ax.plot(valid["epoch"], valid["f1"], label="Faster R-CNN",
                color=COLORS["Faster R-CNN"], linewidth=2, linestyle="--", marker="s", markersize=4)

ax.set_xlabel("Epoch")
ax.set_ylabel("F1-Score")
ax.set_title("F1-Score Evolution (Val)", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("f1_evolution.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Visual Inference — YOLO Models

Side-by-side detection examples on test images.

In [ ]:
try:
    from ultralytics import YOLO as UltralyticsYOLO
    ULTRALYTICS_OK = True
except ImportError:
    ULTRALYTICS_OK = False
    print("ultralytics not installed — skipping YOLO inference visualisation")

COLORS_BGR = [
    (255, 56,  56),  # Button     - red
    (255, 157, 151), # Text       - salmon
    (255, 112, 31),  # Image      - orange
    (255, 178, 29),  # Icon       - yellow
    (207, 210, 49),  # Input      - lime
    (72,  249, 10),  # Link       - green
    (146, 204, 23),  # Checkbox   - olive
    (61,  219, 134), # Toggle     - mint
    (26,  147, 52),  # Toolbar    - dark green
    (0,   212, 187), # Navigation - teal
    (44,  153, 168), # Modal      - cyan
    (0,   194, 255), # Tab        - sky blue
]

def plot_yolo_detections(model_name, run_dir, test_images_dir, n_images=3):
    """Run YOLO inference on n_images and return annotated images."""
    if not ULTRALYTICS_OK:
        return []
    weights = os.path.join(run_dir, "weights/best.pt")
    if not os.path.isfile(weights):
        print(f"  {model_name}: weights not found at {weights}")
        return []

    model = UltralyticsYOLO(weights)
    imgs  = sorted([f for f in os.listdir(test_images_dir)
                    if f.lower().endswith((".png", ".jpg", ".jpeg"))])[:n_images]
    results_list = []
    for fname in imgs:
        img_path = os.path.join(test_images_dir, fname)
        res = model(img_path, verbose=False, conf=0.25)[0]
        results_list.append(res.plot())   # returns BGR numpy array
    return results_list


if ULTRALYTICS_OK and os.path.isdir(TEST_IMAGES):
    import cv2
    N = 3
    fig, axes = plt.subplots(len(YOLO_RUNS), N, figsize=(18, 6 * len(YOLO_RUNS)))
    fig.suptitle("YOLO Detections on Test Images (conf ≥ 0.25)", fontsize=14, fontweight="bold")

    for row_idx, (name, run_dir) in enumerate(YOLO_RUNS.items()):
        annotated = plot_yolo_detections(name, run_dir, TEST_IMAGES, n_images=N)
        for col_idx in range(N):
            ax = axes[row_idx, col_idx] if len(YOLO_RUNS) > 1 else axes[col_idx]
            if col_idx < len(annotated):
                img_rgb = cv2.cvtColor(annotated[col_idx], cv2.COLOR_BGR2RGB)
                ax.imshow(img_rgb)
            else:
                ax.text(0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            if col_idx == 0:
                ax.set_ylabel(name, fontsize=12, rotation=90, labelpad=10)

    plt.tight_layout()
    plt.savefig("yolo_detections.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: yolo_detections.png")
else:
    print("Skipping YOLO inference (ultralytics not available or test folder missing)")


## 9. Visual Inference — Faster R-CNN

Run only after training completes (`best_fasterrcnn.pth` must exist).

In [ ]:
import torch, torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF
import matplotlib.patches as patches

def load_frcnn_model(weights_path, num_classes=13):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    return model


def draw_frcnn(ax, image_pil, output, class_names, score_thresh=0.4):
    ax.imshow(image_pil)
    boxes  = output["boxes"].numpy()
    labels = output["labels"].numpy()
    scores = output["scores"].numpy()
    cmap   = plt.cm.get_cmap("tab20", len(class_names))
    for box, label, score in zip(boxes, labels, scores):
        if score < score_thresh:
            continue
        x1, y1, x2, y2 = box
        cls = class_names[label - 1] if 1 <= label <= len(class_names) else str(label)
        color = cmap(label - 1)
        rect  = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=1.5, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 2, f"{cls} {score:.2f}", fontsize=6,
                color="white", backgroundcolor=color)
    ax.axis("off")


if os.path.isfile(FRCNN_PTH) and os.path.isdir(TEST_IMAGES):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    frcnn  = load_frcnn_model(FRCNN_PTH).to(device)

    test_files = sorted([f for f in os.listdir(TEST_IMAGES)
                         if f.lower().endswith((".png", ".jpg", ".jpeg"))])[:4]

    fig, axes = plt.subplots(1, len(test_files), figsize=(5 * len(test_files), 5))
    fig.suptitle("Faster R-CNN Detections (conf ≥ 0.4)", fontsize=13, fontweight="bold")

    with torch.no_grad():
        for ax, fname in zip(axes, test_files):
            img_pil = Image.open(os.path.join(TEST_IMAGES, fname)).convert("RGB")
            tensor  = TF.to_tensor(img_pil).unsqueeze(0).to(device)
            out     = frcnn(tensor)[0]
            out     = {k: v.cpu() for k, v in out.items()}
            draw_frcnn(ax, img_pil, out, CLASS_NAMES)

    plt.tight_layout()
    plt.savefig("frcnn_detections.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: frcnn_detections.png")
else:
    print("Faster R-CNN weights not found — run this cell after training completes.")


## 10. Summary Table (Print-Ready)

In [ ]:
print("=" * 70)
print(f"{'Model':<15} {'Precision':>10} {'Recall':>10} {'F1':>10} {'mAP@50':>10} {'mAP@50-95':>10}")
print("-" * 70)

for model_name, row in summary_df.iterrows():
    def fmt(v): return f"{v:.4f}" if isinstance(v, float) else str(v)
    print(f"{model_name:<15} {fmt(row.get('Precision',float('nan'))):>10} "
          f"{fmt(row.get('Recall',float('nan'))):>10} "
          f"{fmt(row.get('F1-Score',float('nan'))):>10} "
          f"{fmt(row.get('mAP@50',float('nan'))):>10} "
          f"{fmt(row.get('mAP@50-95',float('nan'))):>10}")
print("=" * 70)
print("\nNote: Faster R-CNN trained on 5% subset (computational constraints)")
print("      YOLO models trained on full dataset (84,104 images)")
